# Figure 5 — crossing positions by Alfvén Mach number, rebuilt

Two things change from the submitted figure, and nothing else:

1. **The upstream parameters come from `omni2_cache.pkl`**, the same cache the renderings and
   Figure 4 now use, instead of `mach_per_crossing.csv`, which was built from the older
   `omni_mach_cache.pkl` with a ±30 minute match. That old file is what gave the "592 of 635"
   in the drafted G6 answer.
2. **The crossing positions are pressure-normalized** to the reference conditions of the Merka
   et al. (2005) coefficients, $n_a = 7.0$ cm$^{-3}$ and $v_a = 457.5$ km s$^{-1}$, using the
   $P^{-1/6}$ scaling of the standoff distance. The submitted figure rotated into GPE but did not
   normalize, while the caption already claimed it did.

Everything else — the Merka coefficients, the equal-count Mach quartiles, the geotail criterion,
the binned means, the panel layout — is the existing code in
`Bow_Shock_Stats/plot_bs_positions_by_mach.py`, imported rather than reimplemented, so the figure
keeps its appearance and only the data behind it moves.

Cell 5 prints every number the G6 answer needs, before and after each change, so the effect of each
is separable.

In [ ]:
import sys
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

import json

# Machine-specific locations come from local_paths.json beside this notebook. That file is not part of the
# repository: copy local_paths.example.json and fill it in. Without it, BOW_SHOCK is the folder above this one.
_local_file = Path("local_paths.json")
LOCAL = json.loads(_local_file.read_text(encoding="utf-8")) if _local_file.exists() else {}
BOW_SHOCK = Path(LOCAL.get("bow_shock", Path.cwd().parent))
STATS     = BOW_SHOCK / "Bow_Shock_Stats"
MAGCARTO  = BOW_SHOCK / "MagCarto"

OMNI_CACHE = Path(LOCAL.get("omni_cache", BOW_SHOCK / "OMNI" / "omni2_cache.pkl"))
CROSSINGS  = BOW_SHOCK / "bs_crossings_V02.xlsx"
OLD_MACH   = STATS / "mach_per_crossing.csv"          # read only, for the comparison
NEW_MACH   = MAGCARTO / "mach_per_crossing_v2.csv"    # written by this notebook

OUT_FIG = MAGCARTO / "bs_positions_geotail_eqbins_GPE_4panel_v2.png"
OUT_CSV = MAGCARTO / "crossing_positions_gpe_v3.csv"   # v3 = v2 columns + R and tail_fixed (section 5)

# --- choices ------------------------------------------------------------------------------------
OMNI_WINDOW_H   = 4.0     # same window as the renderings and Figure 4
NORMALIZE       = True    # scale positions to the model's reference pressure
N_MACH_BINS     = 4
N_X_BINS        = 12      # bins along X for the dayside mean line

# --- reference conditions of the Merka et al. (2005) GPE coefficients -----------------------------
N_REF_CM3   = 7.0
V_REF_KM_S  = 457.5
PDYN_COEF   = 1.6726e-6   # nPa from cm^-3 and km/s, protons only
P_REF_NPA   = PDYN_COEF * N_REF_CM3 * V_REF_KM_S ** 2

for p in (OMNI_CACHE, CROSSINGS, OLD_MACH):
    print(f"{'ok ' if p.exists() else 'MISSING':<8} {p}")
print(f"{'->':<8} {OUT_FIG}")
print(f"\nreference pressure {P_REF_NPA:.3f} nPa  (n = {N_REF_CM3} cm^-3, v = {V_REF_KM_S} km/s)")
print(f"OMNI window +-{OMNI_WINDOW_H} h, normalisation {'on' if NORMALIZE else 'off'}")

## 1. Per-crossing upstream parameters from the new OMNI cache

In [ ]:
cache = pickle.load(open(OMNI_CACHE, "rb"))
FIELDS = ["Alfven_mach_number", "Plasma_bulk_speed", "Ion_density", "Flow_pressure"]
omni = pd.DataFrame({c: np.asarray(cache[c], dtype=float) for c in FIELDS})
omni["Epoch"] = pd.to_datetime(list(cache["Epoch"]))
omni = omni.dropna(subset=FIELDS).sort_values("Epoch").reset_index(drop=True)
_ep = omni.Epoch.values
print(f"OMNI: {len(omni)} hours with Mach number, speed, density and pressure all valid")

x = pd.read_excel(CROSSINGS, sheet_name=0)
x["time"] = pd.to_datetime(x["time"])
print(f"crossings: {len(x)}")


def omni_row_at(t0, window_h=OMNI_WINDOW_H):
    """Nearest hour with all four fields valid, or None if none within window_h."""
    k = np.searchsorted(_ep, np.datetime64(t0))
    best = None
    for i in (k - 1, k):
        if 0 <= i < len(omni):
            off = (omni.Epoch[i] - t0).total_seconds() / 3600.0
            if best is None or abs(off) < abs(best[1]):
                best = (i, off)
    if best is None or abs(best[1]) > window_h:
        return None
    i, off = best
    return omni.loc[i], off


rows = []
for _, cr in x.iterrows():
    got = omni_row_at(cr["time"])
    rec = dict(time=cr["time"], pass_id=int(cr["pass"]), sc_direction=cr["SC direction"])
    if got is not None:
        r, off = got
        n_p, v_sw = float(r.Ion_density), float(r.Plasma_bulk_speed)
        rec.update(alfven_mach=float(r.Alfven_mach_number), vsw_km_s=v_sw, n_p_cm3=n_p,
                   pdyn_omni_nPa=float(r.Flow_pressure),
                   pdyn_nPa=PDYN_COEF * n_p * v_sw ** 2,
                   omni_dt_minutes=off * 60.0)
    rows.append(rec)

mach_v2 = pd.DataFrame(rows)
mach_v2["pass"] = mach_v2.pass_id          # the plotting module expects a 'pass' column
mach_v2.to_csv(NEW_MACH, index=False)

old = pd.read_csv(OLD_MACH, parse_dates=["time"])
print(f"\n{'':22s}{'old cache':>12s}{'new cache':>12s}")
print(f"{'crossings with M_A':22s}{old.alfven_mach.notna().sum():>12d}{mach_v2.alfven_mach.notna().sum():>12d}")
print(f"{'median |dt| [min]':22s}{old.omni_dt_minutes.abs().median():>12.1f}"
      f"{mach_v2.omni_dt_minutes.abs().median():>12.1f}")
print(f"{'max |dt| [min]':22s}{old.omni_dt_minutes.abs().max():>12.1f}"
      f"{mach_v2.omni_dt_minutes.abs().max():>12.1f}")
print(f"\nwrote {NEW_MACH}")

## 2. Positions, GPE rotation and pressure normalisation

The model, the Merka coefficients, the equal-count binning and the drawing all come from
`plot_bs_positions_by_mach.py`. Only `MACH_CSV` is repointed, so the figure is produced by exactly
the code that produced the submitted one.

In [ ]:
sys.path.insert(0, str(STATS))
import plot_bs_positions_by_mach as bs

bs.MACH_CSV = NEW_MACH                      # the only thing redirected
df_mach = bs.load_mach_crossings()

df_gse_pos = bs.load_positions_gse()
df_gse = bs.match_positions(df_mach, df_gse_pos, "GSE")   # interpolates in time, as before
print(f"matched positions: {len(df_gse)}")

df_gpe_raw = bs.gse_to_gpe(df_gse)
df_gpe = df_gpe_raw.rename(columns={
    "X_RE": "X_RE_GSE", "Y_RE": "Y_RE_GSE", "Z_RE": "Z_RE_GSE",
    "X_RE_GPE": "X_RE",  "Y_RE_GPE": "Y_RE",  "Z_RE_GPE": "Z_RE",
})

In [ ]:
def pressure_scale(pdyn_npa, p_ref=P_REF_NPA):
    """Factor that carries a position measured at pdyn to the reference pressure.

    The standoff distance scales as P^(-1/6), so a boundary observed at a higher pressure sits
    closer in and has to be scaled outward to be compared with the reference-condition model.
    """
    return (pdyn_npa / p_ref) ** (1.0 / 6.0)


df_gpe["pdyn_scale"] = pressure_scale(df_gpe["pdyn_nPa"].to_numpy())

df_gpe["X_RE_raw"] = df_gpe["X_RE"]
df_gpe["Y_RE_raw"] = df_gpe["Y_RE"]
df_gpe["Z_RE_raw"] = df_gpe["Z_RE"]

if NORMALIZE:
    for c in ("X_RE", "Y_RE", "Z_RE"):
        df_gpe[c] = df_gpe[c] * df_gpe["pdyn_scale"]

s = df_gpe["pdyn_scale"]
print(f"pressure scale factor: min {s.min():.3f}, median {s.median():.3f}, max {s.max():.3f}")
print(f"  that is {100 * (s.min() - 1):+.0f}% to {100 * (s.max() - 1):+.0f}% on position, "
      f"interquartile {100 * (s.quantile(.25) - 1):+.0f}% to {100 * (s.quantile(.75) - 1):+.0f}%")
print(f"dynamic pressure: min {df_gpe.pdyn_nPa.min():.2f}, median {df_gpe.pdyn_nPa.median():.2f}, "
      f"max {df_gpe.pdyn_nPa.max():.2f} nPa  (reference {P_REF_NPA:.2f})")
print(f"OMNI's own Flow_pressure differs from n*v^2 by a median of "
      f"{100 * ((df_gpe.pdyn_omni_nPa / df_gpe.pdyn_nPa).median() - 1):+.1f}% (alpha contribution)")

## 3. Mach quartiles, the geotail cut, and every number the G6 answer quotes

In [ ]:
mach_vals = df_mach["alfven_mach"].dropna().to_numpy()
eq_bins, eq_labels, eq_merka = bs.build_equal_count_bins(mach_vals, n_bins=N_MACH_BINS)

print("=" * 72)
print(f"quartile boundaries: {[round(b[1], 1) for b in eq_bins[:-1]]}")
for (lo, hi), lbl in zip(eq_bins, eq_labels):
    m = (df_mach.alfven_mach >= lo) & (df_mach.alfven_mach <= hi)
    print(f"  {lbl:24s} n = {m.sum():3d}   median M_A = {df_mach.loc[m, 'alfven_mach'].median():5.2f}")

# the geotail criterion as the code applies it, on the normalised positions
with bs._override_bin_config(eq_bins, eq_labels, eq_merka):
    xs0, Rs0 = bs._merka_shifted_xr(eq_bins[0])
    x_nose = bs._merka_nose_x(eq_bins[0])
R = np.sqrt(df_gpe.Y_RE ** 2 + df_gpe.Z_RE ** 2)
r_boundary = np.interp(df_gpe.X_RE.to_numpy(), xs0, Rs0, left=np.nan, right=np.nan)
geotail = (df_gpe.X_RE.to_numpy() < 0) & (R.to_numpy() < r_boundary) & np.isfinite(r_boundary)
df_gpe["geotail"] = geotail
print(f"\ngeotail by the translated-surface criterion: {geotail.sum()} crossings "
      f"(model nose of the lowest-Mach bin at X = {x_nose:.1f} R_E)")

# how well a plain cut in X reproduces it
print("\nplain cut in X_GPE, against the translated-surface criterion:")
for cut in (-10, -5, -2, 0):
    plain = df_gpe.X_RE.to_numpy() < cut
    print(f"  X < {cut:>3} R_E: {plain.sum():3d} crossings, "
          f"{int((plain & geotail).sum()):3d} agree, {int((plain ^ geotail).sum()):3d} differ")

In [ ]:
def dayside_mean_table(frame, bins, n_bins=N_X_BINS):
    """Binned dayside mean R(X) per Mach bin, as the figure draws it."""
    out = {}
    for i, (lo, hi) in enumerate(bins):
        m = ((frame.alfven_mach >= lo) & (frame.alfven_mach <= hi)) if i == len(bins) - 1 \
            else ((frame.alfven_mach >= lo) & (frame.alfven_mach < hi))
        day = frame[m & ~frame.geotail]
        if len(day) >= 4:
            xc, mn, _ = bs._binned_stats(day.X_RE.to_numpy(),
                                         np.sqrt(day.Y_RE ** 2 + day.Z_RE ** 2).to_numpy(),
                                         n_bins=n_bins)
            out[(lo, hi)] = (xc, mn)
    return out


# the same thing without the normalisation, to isolate its effect
raw = df_gpe.copy()
for c in ("X_RE", "Y_RE", "Z_RE"):
    raw[c] = raw[c + "_raw"]
R_raw = np.sqrt(raw.Y_RE ** 2 + raw.Z_RE ** 2)
raw["geotail"] = (raw.X_RE.to_numpy() < 0) & \
                 (R_raw.to_numpy() < np.interp(raw.X_RE.to_numpy(), xs0, Rs0,
                                               left=np.nan, right=np.nan)) & \
                 np.isfinite(np.interp(raw.X_RE.to_numpy(), xs0, Rs0, left=np.nan, right=np.nan))

norm_tab = dayside_mean_table(df_gpe, eq_bins)
raw_tab = dayside_mean_table(raw, eq_bins)

print("effect of the pressure normalisation on the dayside mean line, per Mach bin")
print(f"  {'bin':24s}{'median |dR| [R_E]':>20s}{'max |dR|':>12s}")
for bk, lbl in zip(eq_bins, eq_labels):
    if bk in norm_tab and bk in raw_tab:
        xa, ya = norm_tab[bk]
        xb, yb = raw_tab[bk]
        yb_i = np.interp(xa, xb, yb, left=np.nan, right=np.nan)
        d = np.abs(ya - yb_i)
        print(f"  {lbl:24s}{np.nanmedian(d):>20.2f}{np.nanmax(d):>12.2f}")
print(f"\ngeotail count: {int(df_gpe.geotail.sum())} normalised, {int(raw.geotail.sum())} not")

## 4. The figure, with a quadric fitted to each Mach quartile

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# ---- Merka's own form, fitted to our crossings (R2-48) -------------------------------------------
# The model is  a1 x^2 + y^2 + a3 z^2 + 2 a4 x y + 2 a7 x + 2 a8 y + a10 = 0, six free coefficients
# once the y^2 coefficient is fixed at one and north-south symmetry removes the yz, xz and z terms.
#
# They are fitted by minimising the orthogonal distance from each crossing to the surface, not by
# least squares on the value of the polynomial.  The algebraic residual is weighted by the local
# gradient, so crossings far from Earth dominate it; on the lowest Mach quartile that returns a
# surface which closes down the tail and lies further from the crossings than the published model.
# Each point is projected onto the surface by Newton steps along the gradient, and the fit starts
# from Merka's coefficients so the comparison is against the model rather than an arbitrary guess.
COEFFS = ("a1", "a3", "a4", "a7", "a8", "a10")
PROJECTION_STEPS = 25
CURVE_POINTS = 3000
MISSING_PENALTY = 50.0        # charged where a surface the crossing cannot lie on is proposed
FIT_LABEL = "quadric fitted to this bin"


def quadric_value(p, x, y, z):
    return (p["a1"] * x ** 2 + y ** 2 + p["a3"] * z ** 2 + 2 * p["a4"] * x * y
            + 2 * p["a7"] * x + 2 * p["a8"] * y + p["a10"])


def quadric_gradient(p, x, y, z):
    return (2 * p["a1"] * x + 2 * p["a4"] * y + 2 * p["a7"],
            2 * y + 2 * p["a4"] * x + 2 * p["a8"],
            2 * p["a3"] * z)


def orthogonal_distance(p, x0, y0, z0):
    """Signed distance to the nearest point of the surface, positive outside."""
    x, y, z = x0.copy(), y0.copy(), z0.copy()
    for _ in range(PROJECTION_STEPS):
        gx, gy, gz = quadric_gradient(p, x, y, z)
        step = quadric_value(p, x, y, z) / np.maximum(gx ** 2 + gy ** 2 + gz ** 2, 1e-12)
        x, y, z = x - step * gx, y - step * gy, z - step * gz
    d = np.sqrt((x - x0) ** 2 + (y - y0) ** 2 + (z - z0) ** 2)
    return np.where(quadric_value(p, x0, y0, z0) > 0, d, -d)


def fit_quadric(x, y, z, start):
    def residual(v):
        d = orthogonal_distance(dict(zip(COEFFS, v)), x, y, z)
        return np.where(np.isfinite(d), d, MISSING_PENALTY)

    solution = least_squares(residual, np.array([start[c] for c in COEFFS]),
                             method="lm", max_nfev=40000)
    return dict(zip(COEFFS, solution.x))


def quadric_xy(p, x):
    """The X-Y cut, dusk branch: the same cut the Merka curve is drawn as."""
    b = p["a4"] * x + p["a8"]
    disc = b ** 2 - (p["a1"] * x ** 2 + 2 * p["a7"] * x + p["a10"])
    ok = disc >= 0
    return x[ok], -b[ok] + np.sqrt(disc[ok])


def quadric_nose(p):
    """Where the surface crosses the Sun-Earth line: the discriminant in y vanishes."""
    A = p["a4"] ** 2 - p["a1"]
    B = 2.0 * (p["a4"] * p["a8"] - p["a7"])
    C = p["a8"] ** 2 - p["a10"]
    disc = B * B - 4 * A * C
    if disc < 0 or abs(A) < 1e-12:
        return np.nan
    roots = [r for r in ((-B + np.sqrt(disc)) / (2 * A), (-B - np.sqrt(disc)) / (2 * A))
             if 0 < r < 100]
    return min(roots) if roots else np.nan


# ---- the tail flag, with the criterion evaluated over the whole orbit ----------------------------
with bs._override_bin_config(eq_bins, eq_labels, eq_merka):
    xs_cut, Rs_cut = bs._merka_shifted_xr(eq_bins[0])
R_vals   = np.hypot(df_gpe["Y_RE"], df_gpe["Z_RE"])
boundary = np.interp(df_gpe["X_RE"], xs_cut, Rs_cut, left=np.nan, right=np.nan)
is_tail  = (df_gpe["X_RE"] < 0) & (R_vals < boundary) & np.isfinite(boundary)
print(f"reference curve reaches x = {xs_cut.min():.1f} R_E   "
      f"tail crossings {int(is_tail.sum())}   "
      f"crossings still beyond the curve {int((df_gpe['X_RE'] < xs_cut.min()).sum())}")

# ---- one fit per Mach quartile, on the same crossings the binned means use -----------------------
fitted_curves = {}
for i, (lo, hi) in enumerate(eq_bins):
    inside = ((df_gpe["alfven_mach"] >= lo) & (df_gpe["alfven_mach"] <= hi)) \
        if i == len(eq_bins) - 1 \
        else ((df_gpe["alfven_mach"] >= lo) & (df_gpe["alfven_mach"] < hi))
    sub = df_gpe[inside & ~is_tail]
    x = sub["X_RE"].to_numpy()
    y = sub["Y_RE"].to_numpy()
    z = sub["Z_RE"].to_numpy()

    start = bs._interp_merka_coeffs(sub["alfven_mach"].median())
    p = fit_quadric(x, y, z, start)
    fitted_curves[(lo, hi)] = quadric_xy(p, np.linspace(x.min(), x.max(), CURVE_POINTS))

    ours  = np.abs(orthogonal_distance(p, x, y, z))
    model = np.abs(orthogonal_distance(start, x, y, z))
    print(f"  M_A {lo:5.2f}-{hi:5.2f}  n={len(sub):4d}  a1={p['a1']:+.2f}  "
          f"nose={quadric_nose(p):5.2f} (model {quadric_nose(start):5.2f})  "
          f"median distance {np.median(ours):.2f} (model {np.median(model):.2f}) R_E  "
          f"x sampled {x.min():6.1f} to {x.max():5.1f}")

with bs._override_bin_config(eq_bins, eq_labels, eq_merka):
    bs.make_figure_geotail_4panel(df_gpe, "GPE — equal-count bins", OUT_FIG, n_bins=N_X_BINS,
                                  extra_curves=fitted_curves, extra_label=FIT_LABEL)

## 5. Write the per-crossing table

In [ ]:
# R and the tail flag that the fits and the figure in section 4 use (is_tail, criterion evaluated over the
# whole orbit). The repository publishes this table as database_files/crossing_positions_gpe_v3.csv.
df_gpe["R"] = np.hypot(df_gpe["Y_RE"], df_gpe["Z_RE"])
df_gpe["tail_fixed"] = np.asarray(is_tail, dtype=bool)

cols = ["time", "pass", "sc_direction", "alfven_mach", "vsw_km_s", "n_p_cm3",
        "pdyn_nPa", "pdyn_omni_nPa", "omni_dt_minutes", "alpha_deg", "pdyn_scale",
        "X_RE_GSE", "Y_RE_GSE", "Z_RE_GSE",
        "X_RE_raw", "Y_RE_raw", "Z_RE_raw",
        "X_RE", "Y_RE", "Z_RE", "geotail", "R", "tail_fixed"]
have = [c for c in cols if c in df_gpe.columns]
df_gpe[have].to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}  ({len(df_gpe)} rows, {int(df_gpe.tail_fixed.sum())} with tail_fixed)")
print("\nX_RE/Y_RE/Z_RE are GPE and pressure-normalised; *_raw are GPE before normalisation.")